In [1]:
import torch
import time

# ============================================================
# SEPARATE MEAN COMPUTATION
# This simulates what happens in an unfused LayerNorm
# Step 1 only: compute mean
# In unfused approach, this would be its own kernel
# Writing result to memory before next step can begin
# ============================================================

torch.manual_seed(42)

# Simulate a batch of tokens, each with a hidden dimension
# In a real transformer this might be [batch x seq_len x d_model]
# We flatten to [num_tokens x d_model] for clarity
num_tokens = 64      # number of tokens in this batch
d_model = 1024       # hidden dimension size (each token is 1024 numbers)

# Create input tensor — simulates the output of attention or MLP
x = torch.randn(num_tokens, d_model)

print(f"Input shape: {x.shape}")
print(f"Each token has {d_model} numbers")
print(f"We need to compute mean and variance for each token separately")
print(f"That means {num_tokens} separate reductions\n")

# ============================================================
# STEP 1: Compute mean for each token (each row)
# dim=1 means reduce along the d_model dimension
# keepdim=True keeps the shape as [64, 1] instead of [64]
# so we can broadcast it back against the [64, 1024] input
# ============================================================

mean = x.mean(dim=1, keepdim=True)   # shape: [64, 1]

print(f"Mean shape: {mean.shape}")
print(f"First token mean: {mean[0].item():.6f}")
print(f"This is ONE number summarizing all 1024 values of token 0")

# In unfused approach: mean would be written to HBM here
# Next kernel would read x from HBM again + read mean from HBM
# That's 2 HBM reads just to start computing variance

Input shape: torch.Size([64, 1024])
Each token has 1024 numbers
We need to compute mean and variance for each token separately
That means 64 separate reductions

Mean shape: torch.Size([64, 1])
First token mean: 0.008771
This is ONE number summarizing all 1024 values of token 0


In [2]:
# ============================================================
# STEP 2: Compute variance (separate kernel in unfused approach)
# Must read x from HBM again
# Must read mean from HBM again
# Two memory reads before any useful work begins
# ============================================================

# Variance formula: mean of squared deviations from mean
# For each element: (x - mean)^2
# Then average those squared deviations

# Subtract mean from every element (broadcasting: [64,1024] - [64,1] = [64,1024])
deviations = x - mean                          # how far each value is from its row's mean
squared_deviations = deviations ** 2           # square to make all positive
variance = squared_deviations.mean(dim=1, keepdim=True)   # average the squared deviations

print(f"Deviations shape: {deviations.shape}")
print(f"Variance shape: {variance.shape}")
print(f"First token variance: {variance[0].item():.6f}")

# Standard deviation is just sqrt of variance
std = torch.sqrt(variance + 1e-5)   # add epsilon to prevent division by zero
print(f"First token std: {std[0].item():.6f}")

# In unfused approach: variance written to HBM here
# Next kernel reads x + mean + variance from HBM
# That's 3 HBM reads for the normalization step

Deviations shape: torch.Size([64, 1024])
Variance shape: torch.Size([64, 1])
First token variance: 1.009132
First token std: 1.004560


In [3]:
# ============================================================
# STEP 3+4: Normalize then rescale (still separate in unfused)
# This would be 2 more kernel launches in a naive implementation
# ============================================================

epsilon = 1e-5

# Normalize: subtract mean, divide by std
# Each token's 1024 values now have mean≈0, std≈1
x_normalized = (x - mean) / std

print(f"Normalized shape: {x_normalized.shape}")
print(f"\nBefore normalization — token 0:")
print(f"  mean={mean[0].item():.4f}, std={std[0].item():.4f}")
print(f"  first 5 values: {x[0, :5].tolist()}")

print(f"\nAfter normalization — token 0:")
norm_mean = x_normalized[0].mean().item()
norm_std = x_normalized[0].std().item()
print(f"  mean={norm_mean:.6f} (should be ~0)")
print(f"  std={norm_std:.6f}  (should be ~1)")
print(f"  first 5 values: {[round(v,4) for v in x_normalized[0, :5].tolist()]}")

# ============================================================
# STEP 4: Apply learned parameters gamma and beta
# gamma starts at 1.0 (no scaling initially)
# beta starts at 0.0 (no shift initially)
# These are learned during training
# ============================================================

gamma = torch.ones(d_model)    # shape: [1024] — one scale per hidden dimension
beta = torch.zeros(d_model)    # shape: [1024] — one shift per hidden dimension

# Apply rescaling
x_layernorm = gamma * x_normalized + beta

print(f"\nAfter gamma/beta rescaling: {x_layernorm.shape}")
print(f"(With gamma=1, beta=0 this equals normalized — same values)")
print(f"In a trained model, gamma and beta would be non-trivial learned values")

Normalized shape: torch.Size([64, 1024])

Before normalization — token 0:
  mean=0.0088, std=1.0046
  first 5 values: [1.9269152879714966, 1.4872840642929077, 0.9007171988487244, -2.1055209636688232, 0.6784184575080872]

After normalization — token 0:
  mean=0.000000 (should be ~0)
  std=1.000484  (should be ~1)
  first 5 values: [1.9094, 1.4718, 0.8879, -2.1047, 0.6666]

After gamma/beta rescaling: torch.Size([64, 1024])
(With gamma=1, beta=0 this equals normalized — same values)
In a trained model, gamma and beta would be non-trivial learned values


In [6]:
# ============================================================
# FUSED LAYERNORM
# PyTorch's built-in LayerNorm IS a fused kernel under the hood
# It does everything in one pass — no separate HBM writes between steps
# We compare it against our manual unfused version
# ============================================================

import torch
import torch.nn as nn
import time

torch.manual_seed(42)

num_tokens = 64
d_model = 1024
x = torch.randn(num_tokens, d_model)

# ============================================================
# METHOD 1: Manual unfused LayerNorm (step by step)
# Simulates separate kernel launches with intermediate writes
# ============================================================

def manual_layernorm_unfused(x, gamma, beta, eps=1e-5):
    # Step 1: compute mean — separate pass over data
    mean = x.mean(dim=-1, keepdim=True)

    # Step 2: compute variance — another pass over data
    variance = ((x - mean) ** 2).mean(dim=-1, keepdim=True)

    # Step 3: normalize — another pass
    x_norm = (x - mean) / torch.sqrt(variance + eps)

    # Step 4: rescale — another pass
    return gamma * x_norm + beta

# ============================================================
# METHOD 2: PyTorch fused LayerNorm
# torch.nn.LayerNorm calls optimized CUDA kernels
# All steps happen in one fused pass
# ============================================================

layernorm = nn.LayerNorm(d_model)   # creates fused LayerNorm layer
gamma = layernorm.weight            # the gamma parameter
beta = layernorm.bias               # the beta parameter

# ============================================================
# VERIFY both give same results
# ============================================================

out_manual = manual_layernorm_unfused(x, gamma, beta)
out_fused = layernorm(x)

match = torch.allclose(out_manual, out_fused, atol=1e-5)
print(f"Manual unfused output shape: {out_manual.shape}")
print(f"Fused output shape:          {out_fused.shape}")
print(f"Results match: {match}")
print(" Both give identical outputs — just different speeds\n")

# ============================================================
# TIME COMPARISON: unfused vs fused
# ============================================================

num_runs = 1000

# Time the manual unfused version
torch.cuda.synchronize() if torch.cuda.is_available() else None
start = time.time()
for _ in range(num_runs):
    out = manual_layernorm_unfused(x, gamma, beta)
unfused_time = (time.time() - start) / num_runs * 1000

# Time the fused version
torch.cuda.synchronize() if torch.cuda.is_available() else None
start = time.time()
for _ in range(num_runs):
    out = layernorm(x)
fused_time = (time.time() - start) / num_runs * 1000

print(f"Unfused LayerNorm time: {unfused_time:.4f} ms")
print(f"Fused LayerNorm time:   {fused_time:.4f} ms")
print(f"Speedup from fusion:    {unfused_time/fused_time:.2f}x")

Manual unfused output shape: torch.Size([64, 1024])
Fused output shape:          torch.Size([64, 1024])
Results match: True
 Both give identical outputs — just different speeds

Unfused LayerNorm time: 0.2160 ms
Fused LayerNorm time:   0.0440 ms
Speedup from fusion:    4.91x


In [5]:
# ============================================================
# EXPERIMENT: measure fused vs unfused across different sizes
# This shows how fusion helps more as d_model grows
# Because bigger tensors = more memory traffic = fusion saves more
# ============================================================

import torch
import torch.nn as nn
import time

def manual_layernorm_unfused(x, gamma, beta, eps=1e-5):
    mean = x.mean(dim=-1, keepdim=True)
    variance = ((x - mean) ** 2).mean(dim=-1, keepdim=True)
    x_norm = (x - mean) / torch.sqrt(variance + eps)
    return gamma * x_norm + beta

configs = [
    (64,  256),    # small: 64 tokens, 256 hidden dim
    (64,  512),
    (64,  1024),   # typical BERT-size
    (64,  2048),
    (64,  4096),   # typical large model hidden dim
    (64,  8192),   # large model
]

num_runs = 500

print(f"{'d_model':<10} {'Unfused (ms)':<18} {'Fused (ms)':<18} {'Speedup'}")
print("-" * 60)

for num_tokens, d_model in configs:
    x = torch.randn(num_tokens, d_model)
    ln = nn.LayerNorm(d_model)
    gamma = ln.weight
    beta = ln.bias

    # Time unfused
    start = time.time()
    for _ in range(num_runs):
        out = manual_layernorm_unfused(x, gamma, beta)
    unfused_ms = (time.time() - start) / num_runs * 1000

    # Time fused
    start = time.time()
    for _ in range(num_runs):
        out = ln(x)
    fused_ms = (time.time() - start) / num_runs * 1000

    speedup = unfused_ms / fused_ms
    print(f"{d_model:<10} {unfused_ms:<18.4f} {fused_ms:<18.4f} {speedup:.2f}x")

print("\n📌 What to observe:")
print("As d_model grows, the fused version saves proportionally more time")
print("Because bigger tensors have more memory traffic to eliminate")
print("At 8192 hidden dim (105B model territory), fusion becomes critical")

d_model    Unfused (ms)       Fused (ms)         Speedup
------------------------------------------------------------
256        0.1213             0.0227             5.34x
512        0.1406             0.0288             4.88x
1024       0.5527             0.2651             2.08x
2048       0.5348             0.0862             6.21x
4096       0.9054             0.1603             5.65x
8192       1.7522             0.3635             4.82x

📌 What to observe:
As d_model grows, the fused version saves proportionally more time
Because bigger tensors have more memory traffic to eliminate
At 8192 hidden dim (105B model territory), fusion becomes critical
